# STT Word-Alignment — Decode audio-driven timing

The choreography anchors every reveal to a **word index** in the beat's narration (`atWordIndex`). To place that reveal on the real timeline it needs a *time* for each word.

- **even_split** — spreads words on equal slices. An estimate; a quick clause and a slow one get the same time, so the animation drifts off the words.
- **align_words_to_tokens** — takes an STT transcript (word + start/end) and maps it back onto the narration's OWN tokens (difflib align + interpolate the gaps). Each token keeps its index, and lands on when it's actually spoken.

This notebook runs the real `decode.timing` functions the pipeline uses.

In [1]:
import os, sys

# Find the backend package (apps/backend) by walking up from the notebook.
d = os.getcwd()
while d != os.path.dirname(d):
    cand = os.path.join(d, 'apps', 'backend')
    if os.path.isdir(os.path.join(cand, 'decode')):
        sys.path.insert(0, cand); break
    d = os.path.dirname(d)

from decode.timing import align_words_to_tokens, even_split_words, Word
print('loaded decode.timing from', cand)

loaded decode.timing from /Users/moinuddinshaik/Downloads/decode/.claude/worktrees/linear-twirling-clover/apps/backend


## 1. A narration line and its even-split timing

This is what shipped before: every word an equal slice of the clip.

In [2]:
NARRATION = "The loss starts high, then each step lowers it until the loss is low."
DURATION = 6.0  # measured length of the spoken clip, in seconds

def show(words):
    for i, w in enumerate(words):
        print(f"{i:2}  {w.text:10} {w.start:5.2f} → {w.end:5.2f}")

print('EVEN SPLIT (estimate)')
show(even_split_words(NARRATION, DURATION))

EVEN SPLIT (estimate)
 0  The         0.00 →  0.43
 1  loss        0.43 →  0.86
 2  starts      0.86 →  1.29
 3  high,       1.29 →  1.71
 4  then        1.71 →  2.14
 5  each        2.14 →  2.57
 6  step        2.57 →  3.00
 7  lowers      3.00 →  3.43
 8  it          3.43 →  3.86
 9  until       3.86 →  4.29
10  the         4.29 →  4.71
11  loss        4.71 →  5.14
12  is          5.14 →  5.57
13  low.        5.57 →  6.00


## 2. An STT transcript, and the aligned timing

Real speech-to-text (e.g. Whisper `timestamp_granularities=["word"]`) returns `(word, start, end)` — often dropping filler and stripping punctuation. Here's a realistic transcript where the opening lingers and the middle is quick. `align_words_to_tokens` maps it back onto the narration's own tokens.

In [3]:
# (word, start, end) as an STT model would return it — note: no 'the', no punctuation,
# uneven pacing (slow start, fast middle, held final 'low').
STT = [
    ('loss', 0.35, 0.85), ('starts', 0.85, 1.35), ('high', 1.35, 2.20),
    ('then', 2.60, 2.85), ('each', 2.85, 3.05), ('step', 3.05, 3.30),
    ('lowers', 3.30, 3.70), ('it', 3.70, 3.85), ('until', 3.95, 4.30),
    ('loss', 4.55, 4.95), ('is', 4.95, 5.10), ('low', 5.40, 6.00),
]

aligned = align_words_to_tokens(NARRATION, STT, DURATION)
print('STT-ALIGNED (real timing, mapped onto narration tokens)')
show(aligned)
print('\none Word per narration token:', [w.text for w in aligned] == NARRATION.split())

STT-ALIGNED (real timing, mapped onto narration tokens)
 0  The         0.00 →  0.35
 1  loss        0.35 →  0.85
 2  starts      0.85 →  1.35
 3  high,       1.35 →  2.60
 4  then        2.60 →  2.85
 5  each        2.85 →  3.05
 6  step        3.05 →  3.30
 7  lowers      3.30 →  3.70
 8  it          3.70 →  3.95
 9  until       3.95 →  4.25
10  the         4.25 →  4.55
11  loss        4.55 →  4.95
12  is          4.95 →  5.40
13  low.        5.40 →  6.00

one Word per narration token: True


## 3. See the drift: even-split vs STT-aligned

Each word plotted at its **start** time. Where the two rows diverge, the even-split animation would fire early or late.

In [4]:
even = even_split_words(NARRATION, DURATION)

try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(12, 3))
    for row, (label, words, color) in enumerate([
        ('STT-aligned', aligned, '#1a936f'),
        ('even-split',  even,    '#b0663b'),
    ]):
        y = 1 - row
        for w in words:
            ax.plot([w.start, w.end], [y, y], color=color, lw=6, solid_capstyle='butt', alpha=0.35)
            ax.text(w.start, y + 0.08, w.text, fontsize=8, rotation=45, va='bottom', ha='left')
        ax.text(-0.15, y, label, ha='right', va='center', fontweight='bold', color=color)
    ax.set_ylim(-0.6, 1.8); ax.set_yticks([])
    ax.set_xlabel('seconds'); ax.set_title('Word timing: STT-aligned vs even-split')
    ax.spines[['top', 'left', 'right']].set_visible(False)
    plt.tight_layout(); plt.show()
except ModuleNotFoundError:
    # No matplotlib in this kernel — fall back to a text timeline of word onsets.
    W = 60
    def bar(words):
        cells = [' '] * W
        for w in words:
            col = min(W - 1, int(w.start / DURATION * W))
            cells[col] = '#'
        return ''.join(cells)
    print('pip install matplotlib for the chart. Text timeline (word onsets):\n')
    print('STT-aligned |' + bar(aligned) + '|')
    print('even-split  |' + bar(even) + '|')
    print(f'             0s{" " * (W - 4)}{DURATION:.0f}s')

pip install matplotlib for the chart. Text timeline (word onsets):

STT-aligned |#  #    #    #            # # # #    # #  #  #   #    #     |
even-split  |#   #   #   #    #   #   #    #   #   #   #    #   #   #    |
             0s                                                        6s


## 4. What the choreography actually does with it

A verb like `{type: 'indicate', targetId: 'high-loss', atWordIndex: 3}` should fire when the word **'high,'** (index 3) is spoken. Below: three verbs resolve to a *different* frame under each timing. At 24fps that's the frame the animation triggers.

In [5]:
FPS = 24
tokens = NARRATION.split()
for idx in (3, 7, 12):
    token = tokens[idx]
    t_even = even[idx].start
    t_stt = aligned[idx].start
    print(f"verb @ word {idx:2} ('{token}')  even-split frame {round(t_even*FPS):3}  |  "
          f"STT frame {round(t_stt*FPS):3}  (Δ {round((t_stt-t_even)*FPS):+d} frames)")

verb @ word  3 ('high,')  even-split frame  31  |  STT frame  32  (Δ +2 frames)
verb @ word  7 ('lowers')  even-split frame  72  |  STT frame  79  (Δ +7 frames)
verb @ word 12 ('is')  even-split frame 123  |  STT frame 119  (Δ -5 frames)


## 5. Run real STT on an actual narration clip

This is the exact call `agents/voice` makes. It loads `DECODE_OPENAI_API_KEY` from `apps/backend/.env` (the kernel doesn't auto-load it), picks a real narration mp3 from the local object store, and prints the true word-level timestamp transcript. In the pipeline these words are then aligned onto that clip's own narration via `align_words_to_tokens`."

In [ ]:
import glob

def load_env_key():
    """Return DECODE_OPENAI_API_KEY, loading it from apps/backend/.env if needed."""
    if os.environ.get('DECODE_OPENAI_API_KEY'):
        return os.environ['DECODE_OPENAI_API_KEY']
    envf = os.path.join(cand, '.env')
    if os.path.exists(envf):
        for line in open(envf):
            name, sep, value = line.strip().partition('=')
            if sep and name.startswith('DECODE_OPENAI_'):
                os.environ.setdefault(name, value)
    return os.environ.get('DECODE_OPENAI_API_KEY')

key = load_env_key()

# Point at a specific mp3, or auto-pick the newest narration clip in the store.
AUDIO_PATH = None
if AUDIO_PATH is None:
    clips = sorted(glob.glob(os.path.join(cand, '.data', 'objects', 'narration', '*.mp3')),
                   key=os.path.getmtime, reverse=True)
    AUDIO_PATH = clips[0] if clips else None

if key and AUDIO_PATH:
    from openai import OpenAI
    client = OpenAI(api_key=key, base_url=os.environ.get('DECODE_OPENAI_BASE_URL') or None)
    with open(AUDIO_PATH, 'rb') as audio_file:
        result = client.audio.transcriptions.create(
            model=os.environ.get('DECODE_OPENAI_TRANSCRIBE_MODEL', 'whisper-1'),
            file=audio_file, response_format='verbose_json', timestamp_granularities=['word'])
    real_stt = [(w.word, w.start, w.end) for w in (getattr(result, 'words', None) or [])]
    print(f'transcribed {os.path.basename(AUDIO_PATH)} -> {len(real_stt)} words with timestamps\n')
    for word, start, end in real_stt[:24]:
        print(f'  {word:16} {start:6.2f} -> {end:6.2f}')
    if len(real_stt) > 24:
        print(f'  ... (+{len(real_stt) - 24} more)')
else:
    print('skipped -- no key in apps/backend/.env, or no narration mp3 in the object store')